# Vox — Model Vote Analysis Research Notebook

Analyses the `vox/trade_vote_audit.jsonl` data from QuantConnect ObjectStore.  
Run **Cell 1** first — it tries QC ObjectStore, then a local JSONL path, then
falls back to synthetic demo data so the rest of the notebook always executes.

**Edit `LOCAL_JSONL_PATH` below if you have an exported file.**

In [ ]:
# =====================================================================
# CELL 1 — Imports, config & data loading
# =====================================================================
import os, json, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Config ────────────────────────────────────────────────────────────
# Path to a local exported JSONL file (leave empty to skip)
LOCAL_JSONL_PATH = ""  # e.g. "/path/to/trade_vote_audit.jsonl"

# ObjectStore key used by the strategy
OBJECTSTORE_KEY = "vox/trade_vote_audit.jsonl"

# Round-trip cost assumption (decimal, e.g. 0.002 = 20 bps)
COST = 0.0020

# Vote threshold used by the engine (for 'voted YES' classification)
DEFAULT_VOTE_THR = 0.50

# ── Data loading ──────────────────────────────────────────────────────
def _parse_jsonl(raw_text):
    records = []
    for line in raw_text.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            pass
    return records

def _try_load_objectstore():
    """Try QuantConnect ObjectStore (only works inside QC Research)."""
    try:
        from QuantConnect.Research import QuantBook  # noqa
        qb = QuantBook()
        if qb.object_store.contains_key(OBJECTSTORE_KEY):
            raw = qb.object_store.read(OBJECTSTORE_KEY)
            recs = _parse_jsonl(raw)
            if recs:
                print(f"[load] QC ObjectStore → {len(recs)} records")
                return recs
        print("[load] QC ObjectStore key not found.")
    except Exception as e:
        print(f"[load] QC ObjectStore unavailable: {e}")
    return None

def _try_load_local(path):
    """Try a local JSONL file."""
    if not path:
        return None
    if not os.path.exists(path):
        print(f"[load] Local file not found: {path}")
        return None
    with open(path, 'r') as f:
        recs = _parse_jsonl(f.read())
    print(f"[load] Local file → {len(recs)} records from {path}")
    return recs

def _make_synthetic_records(n_trades=30, seed=42):
    """Generate realistic synthetic data so all analysis cells execute."""
    rng = np.random.default_rng(seed)
    models = ['rf', 'et', 'lgbm_bal', 'hgbc_l2', 'gnb', 'lr_bal', 'xgb_bal', 'cal_et', 'cal_rf']
    symbols = ['ADAUSD', 'SOLUSD', 'ETHUSD', 'BTCUSD', 'LINKUSD', 'DOTUSD']
    modes = ['risk_on_trend', 'chop', 'risk_off']
    records = []
    for i in range(n_trades):
        tid = f"demo_{i:04d}"
        is_win = rng.random() < 0.28  # realistic 28% win rate
        ret = float(rng.normal(0.08, 0.03)) if is_win else float(rng.normal(-0.025, 0.015))
        # active votes — correlated with outcome
        active_votes = {}
        for m in models:
            base = 0.65 if is_win else 0.42
            active_votes[m] = float(np.clip(rng.normal(base, 0.15), 0, 1))
        n_agree = sum(1 for v in active_votes.values() if v >= DEFAULT_VOTE_THR)
        vs = float(np.mean(list(active_votes.values())))
        top3 = float(np.mean(sorted(active_votes.values(), reverse=True)[:3]))
        mode = rng.choice(modes)
        entry_rec = {
            'trade_id': tid, 'entry_type': 'entry', 'symbol': rng.choice(symbols),
            'entry_time': f'2025-{1+i//5:02d}-{1+i%28:02d} 10:00:00',
            'entry_price': float(rng.uniform(1, 100)), 'entry_qty': 10.0,
            'allocation': 0.12, 'risk_profile': 'ruthless_v2', 'ruthless_v2_mode': True,
            'market_mode': mode, 'confirm': rng.choice(['trend', 'chop', None]),
            'entry_path': rng.choice(['apex_path1', 'apex_path2', 'apex_path3']),
            'vote_score': vs, 'dynamic_vote_score': vs * 1.02, 'core_vote_score': vs,
            'vote_yes_fraction': n_agree / len(models),
            'top3_mean': top3,
            'active_model_count': len(models),
            'active_mean': vs, 'active_std': float(np.std(list(active_votes.values()))),
            'active_n_agree': n_agree,
            'pred_return': float(rng.normal(0.03 if is_win else 0.01, 0.02)),
            'ev_score': float(rng.uniform(0, 0.01)),
            'final_score': vs,
            'apex_score': float(rng.uniform(0.3, 0.9)),
            'apex_path': rng.choice(['path1', 'path2', 'path3']),
            'n_agree': n_agree, 'mean_proba': vs, 'cost_bps': 20.0,
            'meta_entry_score': float(rng.uniform(0.3, 0.9)),
            'lane_selected': rng.choice(['scalp', 'continuation', 'runner', None]),
            'active_votes': active_votes,
            'shadow_votes': {m: float(np.clip(rng.normal(0.5, 0.15), 0, 1)) for m in ['et_shallow', 'rf_shallow']},
            'diagnostic_votes': {},
            'effective_model_weights': {m: 1.0 for m in models},
            'v2_opportunity_score': vs,
            'relative_strength_rank': int(rng.integers(1, 6)),
            'scalp_score': float(rng.uniform(0.3, 0.8)),
            'continuation_score': float(rng.uniform(0.3, 0.8)),
            'runner_score': float(rng.uniform(0.3, 0.8)),
        }
        ep = entry_rec['entry_price']
        exit_rec = {
            'trade_id': tid, 'entry_type': 'exit', 'symbol': entry_rec['symbol'],
            'exit_time': f'2025-{1+i//5:02d}-{2+i%28:02d} 10:00:00',
            'exit_price': ep * (1 + ret), 'exit_reason': 'trail' if is_win else 'stop_loss',
            'hold_minutes': int(rng.integers(30, 1440)),
            'realized_return': ret, 'realized_pnl': ret * 1000,
            'fees': 2.0, 'mfe': max(0, ret + 0.01), 'mae': min(0, ret - 0.005),
            'winner': 1 if is_win else 0,
            'entry_time': entry_rec['entry_time'], 'entry_price': ep,
            'risk_profile': 'ruthless_v2', 'ruthless_v2_mode': True,
            'lane_selected': entry_rec['lane_selected'],
            'market_mode': mode,
        }
        records.extend([entry_rec, exit_rec])
    print(f"[load] Using synthetic demo data: {n_trades} trades. "
          f"Set LOCAL_JSONL_PATH to use real data.")
    return records

# ── Attempt load ──────────────────────────────────────────────────────
_records = _try_load_objectstore()
if _records is None:
    _records = _try_load_local(LOCAL_JSONL_PATH)
if _records is None:
    _records = _make_synthetic_records(n_trades=50)

# ── Split entries / exits ─────────────────────────────────────────────
entries_list = [r for r in _records if r.get('entry_type') == 'entry']
exits_list   = [r for r in _records if r.get('entry_type') == 'exit']

entries = pd.DataFrame(entries_list) if entries_list else pd.DataFrame()
exits   = pd.DataFrame(exits_list)   if exits_list   else pd.DataFrame()

print(f"Entry records : {len(entries)}")
print(f"Exit records  : {len(exits)}")

# ── Build paired (closed-trade) DataFrame ────────────────────────────
if len(entries) and len(exits) and 'trade_id' in entries.columns and 'trade_id' in exits.columns:
    closed = exits.merge(
        entries[[
            'trade_id', 'vote_score', 'dynamic_vote_score', 'vote_yes_fraction',
            'top3_mean', 'active_mean', 'active_std', 'active_n_agree', 'n_agree',
            'pred_return', 'ev_score', 'final_score', 'apex_score', 'apex_path',
            'mean_proba', 'market_mode', 'confirm', 'entry_path', 'lane_selected',
            'active_votes', 'shadow_votes', 'diagnostic_votes', 'effective_model_weights',
            'meta_entry_score', 'v2_opportunity_score',
            'scalp_score', 'continuation_score', 'runner_score',
        ]], on='trade_id', how='inner', suffixes=('', '_entry')
    )
else:
    closed = pd.DataFrame()

if 'realized_return' in closed.columns:
    closed['realized_return'] = pd.to_numeric(closed['realized_return'], errors='coerce')
    closed['winner'] = (closed['realized_return'] > 0).astype(int)
    closed = closed.dropna(subset=['realized_return'])

print(f"Closed (paired entry+exit): {len(closed)}")
if len(closed):
    wr = closed['winner'].mean()
    avg_ret = closed['realized_return'].mean()
    print(f"Win rate: {wr:.1%}   Avg return: {avg_ret:.4f}   "
          f"Expectancy: {wr*closed.loc[closed.winner==1,'realized_return'].mean():.4f} / "
          f"{(1-wr)*closed.loc[closed.winner==0,'realized_return'].mean():.4f}")

---
## Cell 2 — Quick summary stats

In [ ]:
# =====================================================================
# CELL 2 — Summary stats for the closed-trade universe
# =====================================================================
if closed.empty:
    print("No closed trades available.")
else:
    rets = closed['realized_return']
    wins = closed[closed['winner'] == 1]
    losses = closed[closed['winner'] == 0]

    summary = {
        'Total trades':       len(closed),
        'Win rate':           f"{closed['winner'].mean():.1%}",
        'Avg win':            f"{wins['realized_return'].mean():.4f}" if len(wins) else 'n/a',
        'Avg loss':           f"{losses['realized_return'].mean():.4f}" if len(losses) else 'n/a',
        'Max win':            f"{rets.max():.4f}",
        'Max loss':           f"{rets.min():.4f}",
        'Net return (sum)':   f"{rets.sum():.4f}",
        'Profit factor':      f"{wins['realized_return'].sum() / abs(losses['realized_return'].sum()):.2f}"
                              if len(losses) and losses['realized_return'].sum() != 0 else 'inf',
        'Sharpe (raw ret)':   f"{rets.mean() / rets.std():.3f}" if rets.std() else 'n/a',
    }
    for k, v in summary.items():
        print(f"  {k:<25} {v}")

    print("\nReturn distribution:")
    print(pd.cut(rets, bins=[-1,-0.05,-0.02,0,0.02,0.05,1]).value_counts().sort_index())

---
## Cell 3 — Per-model discrimination (AUC, win rate, EV)

In [ ]:
# =====================================================================
# CELL 3 — Per-model discrimination at multiple thresholds
# =====================================================================
from sklearn.metrics import roc_auc_score

def _expand_votes(df):
    """Flatten active/shadow/diagnostic vote dicts into per-row Series."""
    rows = []
    for _, row in df.iterrows():
        base = {
            'trade_id':         row.get('trade_id'),
            'realized_return':  row.get('realized_return'),
            'winner':           row.get('winner'),
        }
        for vcol in ('active_votes', 'shadow_votes', 'diagnostic_votes'):
            vdict = row.get(vcol)
            if not isinstance(vdict, dict):
                continue
            for mid, proba in vdict.items():
                rows.append({
                    **base,
                    'model_id':   mid,
                    'vote_source': vcol.replace('_votes', ''),
                    'proba':       float(proba) if proba is not None else np.nan,
                })
    return pd.DataFrame(rows)

def model_report(frame, thresholds=(0.45, 0.50, 0.55, 0.60, 0.65, 0.70)):
    if frame.empty or 'proba' not in frame.columns:
        return pd.DataFrame()
    reports = []
    for mid, grp in frame.groupby('model_id'):
        grp = grp.dropna(subset=['proba', 'realized_return'])
        if len(grp) < 3:
            continue
        y   = grp['winner'].astype(int)
        p   = grp['proba'].astype(float)
        rec = {
            'model': mid,
            'source': grp['vote_source'].iloc[0],
            'n': len(grp),
            'mean_vote': p.mean(),
            'std_vote':  p.std(),
            'auc': np.nan,
            'corr_ret': p.corr(grp['realized_return']),
        }
        try:
            if y.nunique() == 2:
                rec['auc'] = roc_auc_score(y, p)
        except Exception:
            pass
        for t in thresholds:
            yes = grp[p >= t]
            no  = grp[p <  t]
            rec[f'n_yes@{t:.2f}']     = len(yes)
            rec[f'wr_yes@{t:.2f}']    = yes['winner'].mean() if len(yes) else np.nan
            rec[f'avgret_yes@{t:.2f}']= yes['realized_return'].mean() if len(yes) else np.nan
            rec[f'ev_net_yes@{t:.2f}']= (yes['realized_return'].mean() - COST) if len(yes) else np.nan
            rec[f'wr_no@{t:.2f}']     = no['winner'].mean() if len(no) else np.nan
            rec[f'avgret_no@{t:.2f}'] = no['realized_return'].mean() if len(no) else np.nan
        reports.append(rec)
    return pd.DataFrame(reports).sort_values('auc', ascending=False)

if closed.empty:
    print("No data.")
    model_df = pd.DataFrame()
else:
    vote_long = _expand_votes(closed)
    model_df  = model_report(vote_long)
    print(f"Models analysed: {len(model_df)}")
    core_cols = ['model','source','n','auc','mean_vote','std_vote','corr_ret',
                 'n_yes@0.50','wr_yes@0.50','avgret_yes@0.50','ev_net_yes@0.50']
    display_cols = [c for c in core_cols if c in model_df.columns]
    print("\n=== Model Discrimination (sorted by AUC) ===")
    print(model_df[display_cols].to_string(index=False))

---
## Cell 4 — Winners vs Losers (promote / demote recommendations)

In [ ]:
# =====================================================================
# CELL 4 — Promote / demote classification
# =====================================================================
if model_df.empty:
    print("No model data.")
else:
    ev_col = 'ev_net_yes@0.50'
    auc_col = 'auc'

    # Winners: AUC >= 0.52 and positive net EV when voting YES
    winners = model_df[
        (model_df[auc_col].fillna(0) >= 0.52) &
        (model_df[ev_col].fillna(-1) > 0)
    ][['model','auc','mean_vote',ev_col,'n_yes@0.50']].copy()
    winners = winners.sort_values(ev_col, ascending=False)

    # Losers: AUC < 0.52 or negative net EV
    losers = model_df[
        (model_df[auc_col].fillna(0) < 0.52) |
        (model_df[ev_col].fillna(-1) <= 0)
    ][['model','auc','mean_vote',ev_col,'n_yes@0.50']].copy()
    losers = losers.sort_values(ev_col, ascending=True)

    print("=== PROMOTE (AUC >= 0.52 AND net EV > 0) — increase weights ===")
    print(winners.to_string(index=False) if len(winners) else "  (none)")

    print("\n=== DEMOTE (AUC < 0.52 OR negative EV) — move to shadow ===")
    print(losers.to_string(index=False) if len(losers) else "  (none)")

---
## Cell 5 — Threshold sweep (`vote_score`, `n_agree`, `top3_mean`, `apex_score`)

In [ ]:
# =====================================================================
# CELL 5 — Threshold sweep to find optimal gate values
# =====================================================================
def threshold_sweep(df, col, grid, label=None):
    """Return a DataFrame: for each threshold, show n/wr/avg_ret/EV_net/PF."""
    label = label or col
    rows = []
    for t in grid:
        sub = df[df[col].notna() & (df[col] >= t)]
        if len(sub) == 0:
            continue
        wins   = sub[sub['realized_return'] > 0]
        losses = sub[sub['realized_return'] < 0]
        pf     = (wins['realized_return'].sum() / abs(losses['realized_return'].sum())
                  if len(losses) and losses['realized_return'].sum() != 0 else np.inf)
        rows.append({
            'threshold': t,
            'n': len(sub),
            'win_rate': sub['winner'].mean(),
            'avg_ret': sub['realized_return'].mean(),
            'ev_net': sub['realized_return'].mean() - COST,
            'profit_factor': round(pf, 3),
        })
    result = pd.DataFrame(rows)
    print(f"\n=== {label} threshold sweep ===")
    print(result.to_string(index=False) if len(result) else "  (no data)")
    return result

if closed.empty:
    print("No closed trades.")
else:
    sweep_results = {}
    for col, grid, lbl in [
        ('vote_score',        np.arange(0.30, 0.85, 0.05), 'vote_score'),
        ('active_n_agree',    [1, 2, 3, 4, 5, 6],          'n_agree (active models agreeing)'),
        ('top3_mean',         np.arange(0.35, 0.85, 0.05), 'top3_mean'),
        ('apex_score',        np.arange(0.30, 0.80, 0.05), 'apex_score'),
        ('vote_yes_fraction', np.arange(0.20, 0.90, 0.10), 'vote_yes_fraction'),
        ('meta_entry_score',  np.arange(0.30, 0.80, 0.05), 'meta_entry_score'),
        ('ev_score',          np.arange(0.00, 0.012, 0.002), 'ev_score'),
    ]:
        if col in closed.columns:
            closed[col] = pd.to_numeric(closed[col], errors='coerce')
            sweep_results[col] = threshold_sweep(closed, col, grid, lbl)
        else:
            print(f"\n  [{col}] column not found in data")

---
## Cell 6 — Model pair synergies (both vote YES)

In [ ]:
# =====================================================================
# CELL 6 — Pairwise model synergy: which pairs, when BOTH bullish, produce edge?
# =====================================================================
def pairwise_synergy(df, vote_thr=DEFAULT_VOTE_THR, min_n=4):
    if df.empty:
        return pd.DataFrame()
    # Collect all model-level vote series
    all_votes = {}
    for _, row in df.iterrows():
        for vcol in ('active_votes', 'shadow_votes', 'diagnostic_votes'):
            vd = row.get(vcol)
            if isinstance(vd, dict):
                for mid, p in vd.items():
                    if mid not in all_votes:
                        all_votes[mid] = {}
                    try:
                        all_votes[mid][row['trade_id']] = float(p)
                    except (TypeError, ValueError):
                        pass

    # Build a wide DataFrame: trade_id × model
    vote_wide = pd.DataFrame(all_votes).reindex(df.set_index('trade_id').index)
    base = df.set_index('trade_id')[['realized_return', 'winner']]
    combined = base.join(vote_wide)

    model_ids = list(all_votes.keys())
    rows = []
    for a, b in combinations(model_ids, 2):
        if a not in combined.columns or b not in combined.columns:
            continue
        both_yes = combined[(combined[a] >= vote_thr) & (combined[b] >= vote_thr)].dropna(subset=['realized_return'])
        if len(both_yes) < min_n:
            continue
        w  = both_yes['realized_return']
        pf = (w[w>0].sum() / abs(w[w<0].sum())) if (w < 0).any() else np.inf
        rows.append({
            'model_a': a, 'model_b': b,
            'n_both_yes': len(both_yes),
            'win_rate':   both_yes['winner'].mean(),
            'avg_ret':    w.mean(),
            'ev_net':     w.mean() - COST,
            'profit_factor': round(pf, 2),
        })
    return pd.DataFrame(rows).sort_values('ev_net', ascending=False)

if closed.empty:
    print("No data.")
else:
    pairs_df = pairwise_synergy(closed)
    print(f"=== Top model PAIRS (both vote YES >= {DEFAULT_VOTE_THR}) by net EV ===")
    print(pairs_df.head(20).to_string(index=False) if len(pairs_df) else "  (insufficient data for pairs)")

---
## Cell 7 — Regime / market_mode conditional analysis

In [ ]:
# =====================================================================
# CELL 7 — Regime / market_mode analysis
# =====================================================================
if closed.empty:
    print("No data.")
else:
    mode_col = 'market_mode' if 'market_mode' in closed.columns else None

    if mode_col:
        print("=== By market_mode ===")
        grp = closed.groupby(mode_col)['realized_return'].agg(
            n='count', mean='mean', median='median',
            win_rate=lambda s: (s > 0).mean(),
            total='sum',
        ).sort_values('total', ascending=False)
        print(grp.to_string())

    if 'confirm' in closed.columns:
        print("\n=== By confirm path ===")
        grp2 = closed.groupby('confirm')['realized_return'].agg(
            n='count', mean='mean',
            win_rate=lambda s: (s > 0).mean(),
        ).sort_values('mean', ascending=False)
        print(grp2.to_string())

    if 'lane_selected' in closed.columns:
        print("\n=== By lane_selected ===")
        grp3 = closed.groupby('lane_selected')['realized_return'].agg(
            n='count', mean='mean',
            win_rate=lambda s: (s > 0).mean(),
        ).sort_values('mean', ascending=False)
        print(grp3.to_string())

    if 'symbol' in closed.columns:
        print("\n=== By symbol ===")
        grp4 = closed.groupby('symbol')['realized_return'].agg(
            n='count', mean='mean',
            win_rate=lambda s: (s > 0).mean(),
            total='sum',
        ).sort_values('total', ascending=False)
        print(grp4.to_string())

---
## Cell 8 — Profit-weighted ensemble weights & recommendations

In [ ]:
# =====================================================================
# CELL 8 — Compute profit-weighted model weights + save CSV
# =====================================================================
if model_df.empty:
    print("No model data.")
else:
    ev_col  = 'ev_net_yes@0.50'
    auc_col = 'auc'

    w = model_df[['model', auc_col, ev_col, 'mean_vote', 'n']].copy().set_index('model')
    # Weight = max(0, AUC-0.5) × sign(ev_net), floored at 0
    w['raw'] = (
        (w[auc_col] - 0.50).clip(lower=0) *
        np.sign(w[ev_col].fillna(0))
    ).clip(lower=0)
    total_raw = w['raw'].sum()
    w['weight_normalised'] = w['raw'] / total_raw if total_raw > 0 else 0.0

    # Scale to max 2.0 for use in model_registry
    max_w = w['weight_normalised'].max()
    if max_w > 0:
        w['registry_weight'] = (w['weight_normalised'] / max_w * 2.0).round(2)
    else:
        w['registry_weight'] = 1.0

    w_sorted = w.sort_values('weight_normalised', ascending=False)

    print("=== Recommended ensemble weights (models with weight > 0) ===")
    print(w_sorted[w_sorted['weight_normalised'] > 0].to_string())

    print("\n=== Zero-weight models (candidates for shadow/demotion) ===")
    print(w_sorted[w_sorted['weight_normalised'] <= 0][['auc', ev_col]].to_string())

    # Save recommendations
    out_path = "recommended_model_weights.csv"
    w_sorted.reset_index().to_csv(out_path, index=False)
    print(f"\nSaved: {out_path}")

---
## Cell 9 — Visualisations

In [ ]:
# =====================================================================
# CELL 9 — Visualisations
# =====================================================================
if not model_df.empty:
    ev_col  = 'ev_net_yes@0.50'
    auc_col = 'auc'

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot 1: AUC vs. Net EV scatter
    ax = axes[0]
    x = model_df[auc_col].fillna(0.5)
    y = model_df[ev_col].fillna(0) * 100
    colours = ['#2ca02c' if (xi >= 0.52 and yi > 0) else '#d62728' for xi, yi in zip(x, y)]
    ax.scatter(x, y, c=colours, s=80, zorder=3)
    for _, row in model_df.iterrows():
        ax.annotate(row['model'],
                    (row[auc_col] if pd.notna(row[auc_col]) else 0.5,
                     (row[ev_col] if pd.notna(row[ev_col]) else 0) * 100),
                    fontsize=8, ha='center', va='bottom')
    ax.axhline(0, color='grey', lw=0.8, ls='--')
    ax.axvline(0.52, color='grey', lw=0.8, ls='--')
    ax.set_xlabel('AUC')
    ax.set_ylabel('Net EV when YES (%) @ thr=0.50')
    ax.set_title('Model discrimination vs. profitability')

    # Plot 2: Return distribution
    ax2 = axes[1]
    if not closed.empty:
        rets = closed['realized_return']
        ax2.hist(rets[rets < 0],  bins=20, color='#d62728', alpha=0.7, label='Loss')
        ax2.hist(rets[rets >= 0], bins=20, color='#2ca02c', alpha=0.7, label='Win')
        ax2.axvline(rets.mean(), color='navy', lw=1.5, ls='--', label=f'Mean {rets.mean():.4f}')
        ax2.set_xlabel('Realized return')
        ax2.set_ylabel('Count')
        ax2.set_title('Return distribution')
        ax2.legend()

    plt.tight_layout()
    plt.savefig('vote_analysis_charts.png', dpi=120)
    plt.show()
    print("Saved: vote_analysis_charts.png")

---
## Cell 10 — Export vote audit from QC ObjectStore to local JSONL

Run this cell **inside QC Research** to download the audit file for local analysis.
After running, set `LOCAL_JSONL_PATH = "vox_trade_audit_local.jsonl"` in Cell 1.

In [ ]:
# =====================================================================
# CELL 10 — Export ObjectStore → local JSONL  (run inside QC Research)
# =====================================================================
try:
    from QuantConnect.Research import QuantBook
    qb = QuantBook()
    key = "vox/trade_vote_audit.jsonl"
    if qb.object_store.contains_key(key):
        raw = qb.object_store.read(key)
        local_out = "vox_trade_audit_local.jsonl"
        with open(local_out, "w") as f:
            f.write(raw)
        print(f"Saved {len(raw.splitlines())} lines → {local_out}")
        print("Tip: set LOCAL_JSONL_PATH = \"vox_trade_audit_local.jsonl\" in Cell 1 and re-run.")
    else:
        print(f"ObjectStore key '{key}' not found. Run a backtest first.")
except ImportError:
    print("Not in QC Research environment. Skip this cell when running locally.")

---
## How to get real data

### Option A — Run in QuantConnect Research
1. Open Research tab in your QuantConnect project.
2. Upload this `.ipynb` file or paste the cells.
3. Run Cell 10 once to export the ObjectStore audit to a local `.jsonl`.
4. Cell 1 will auto-load from QC ObjectStore — all other cells will use real data.

### Option B — Local JSONL file
1. After running a backtest, the strategy saves `vox/trade_vote_audit.jsonl` to ObjectStore.
2. Export it using Cell 10 (inside QC Research) to get a local copy.
3. Set `LOCAL_JSONL_PATH = "/path/to/vox_trade_audit_local.jsonl"` at the top of Cell 1.
4. Re-run Cell 1 — all subsequent cells will use real data.

### Key audit fields

| Field | Description |
|---|---|
| `trade_id` | Unique ID linking entry + exit |
| `active_votes` | Dict of `{model_id: proba}` for active-role models |
| `shadow_votes` | Dict of `{model_id: proba}` for shadow models |
| `vote_score` | Composite weighted vote score |
| `vote_yes_fraction` | Fraction of active models voting YES |
| `top3_mean` | Mean of top-3 model probabilities |
| `apex_score` | Apex Predator composite score |
| `n_agree` | Count of models with proba >= threshold |
| `market_mode` | `risk_on_trend` / `chop` / `risk_off` |
| `realized_return` | Actual return (exit only) |
| `winner` | 1 = profitable, 0 = loss |
